Dataset

In [2]:

markdown_texts= """
# Healthy Living

## Healthy Diet

A healthy diet includes vegetables, fruits, whole grains,
proteins, and sufficient water.

### Benefits

A balanced diet can support overall health and provide
the nutrients needed by the body.

## Exercise

Regular physical activity can improve fitness and support
a healthy lifestyle.

### Daily Activity

Walking, cycling, swimming, and other physical activities
can be included in a daily routine.

# Travel Tips and Adventure

## Travel Planning

Good travel planning includes selecting a destination,
checking transportation, and preparing important documents.

### Before Traveling

Check your passport, tickets, accommodation, weather,
and local transportation before starting your journey.

## Safety Tips

Keep important documents safe and stay aware of your
surroundings while traveling.

### Adventure Travel

Adventure activities can include hiking, camping,
cycling, and exploring new places.
"""

In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
headers_to_split_on = [
    ("#", "header 1"),
    ("##", "header 2"),
    ("###", "header 3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,
)

chunks = splitter.split_text(markdown_texts)


In [4]:
chunks

[Document(metadata={'header 1': 'Healthy Living', 'header 2': 'Healthy Diet'}, page_content='# Healthy Living  \n## Healthy Diet  \nA healthy diet includes vegetables, fruits, whole grains,\nproteins, and sufficient water.'),
 Document(metadata={'header 1': 'Healthy Living', 'header 2': 'Healthy Diet', 'header 3': 'Benefits'}, page_content='### Benefits  \nA balanced diet can support overall health and provide\nthe nutrients needed by the body.'),
 Document(metadata={'header 1': 'Healthy Living', 'header 2': 'Exercise'}, page_content='## Exercise  \nRegular physical activity can improve fitness and support\na healthy lifestyle.'),
 Document(metadata={'header 1': 'Healthy Living', 'header 2': 'Exercise', 'header 3': 'Daily Activity'}, page_content='### Daily Activity  \nWalking, cycling, swimming, and other physical activities\ncan be included in a daily routine.'),
 Document(metadata={'header 1': 'Travel Tips and Adventure', 'header 2': 'Travel Planning'}, page_content='# Travel Tips a

In [5]:
len(chunks)

8

Pinecone: fully manage cloud api,real time filtering, paid version
Pinecone is a vector database designed to store and search embeddings (vectors) efficiently. In your RAG project, it acts as the searchable memory that sits between your documents and your LLM.

In [6]:
import os
import time
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter


from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

from uuid import uuid4
from dotenv import load_dotenv
load_dotenv()

c:\Users\Lenovo\Documents\Gen AI\Agentic_ai\Agentic_AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [7]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("Pinecone key loaded:", bool(PINECONE_API_KEY))
print("Google key loaded:", bool(GOOGLE_API_KEY))

Pinecone key loaded: True
Google key loaded: True


In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
pc=Pinecone(api_key=PINECONE_API_KEY)

In [15]:
import os
from pinecone import Pinecone, ServerlessSpec

# 1. Load API key
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is not set")

# 2. Create Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

# 3. Test authentication
# print(pc.list_indexes())

# 4. Create index if necessary
index_name = "testing"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# 5. Connect to index
index = pc.Index(index_name)

In [20]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

index_name = "testing-vdb"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

index = pc.Index(index_name)

pinecone_vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [21]:
uuids = [str(uuid4()) for _ in range(len(chunks))]
pinecone_vector_store.add_documents(documents=chunks, ids=uuids)

['223f267c-ac04-4fa1-a552-0552afa526d3',
 '4d17f525-2110-4764-8c59-613549d4969a',
 '8746d3a2-a45b-4ee3-bb80-341cedb593a8',
 'bcf43aab-3c7a-47f7-9c11-067f4d9c64c9',
 '479e164a-c19c-4223-9f80-a3112c439794',
 '496407b7-dc9a-4f80-a134-b017a52b952f',
 '52da3616-f7a9-4c8c-893e-7f8e7dd29a1c',
 '9fbb448f-b9bc-4b86-b55c-152df98d9e0c']

In [23]:
results = pinecone_vector_store.similarity_search("healthy diet", k=3)
results

[Document(id='223f267c-ac04-4fa1-a552-0552afa526d3', metadata={'header 1': 'Healthy Living', 'header 2': 'Healthy Diet'}, page_content='# Healthy Living  \n## Healthy Diet  \nA healthy diet includes vegetables, fruits, whole grains,\nproteins, and sufficient water.'),
 Document(id='4d17f525-2110-4764-8c59-613549d4969a', metadata={'header 1': 'Healthy Living', 'header 2': 'Healthy Diet', 'header 3': 'Benefits'}, page_content='### Benefits  \nA balanced diet can support overall health and provide\nthe nutrients needed by the body.'),
 Document(id='8746d3a2-a45b-4ee3-bb80-341cedb593a8', metadata={'header 1': 'Healthy Living', 'header 2': 'Exercise'}, page_content='## Exercise  \nRegular physical activity can improve fitness and support\na healthy lifestyle.')]

In [26]:
results_with_filter = pinecone_vector_store.similarity_search(
    "healthy diet",k=4,filter={"header 2": "Healthy Diet"})
results_with_filter

[Document(id='223f267c-ac04-4fa1-a552-0552afa526d3', metadata={'header 1': 'Healthy Living', 'header 2': 'Healthy Diet'}, page_content='# Healthy Living  \n## Healthy Diet  \nA healthy diet includes vegetables, fruits, whole grains,\nproteins, and sufficient water.'),
 Document(id='4d17f525-2110-4764-8c59-613549d4969a', metadata={'header 1': 'Healthy Living', 'header 2': 'Healthy Diet', 'header 3': 'Benefits'}, page_content='### Benefits  \nA balanced diet can support overall health and provide\nthe nutrients needed by the body.')]

SIMILARITY SEARCH

In [ ]:
results = pinecone_vector_store.similarity_search_with_score("healthy diet",k=3)
for res, score in results:
    print(
        f"* [SIM={score:.4f}] "
        f"{res.metadata.get('header 2', 'No Header 2')} "
        f"- {res.page_content[:50]}"
    )
    #0 score is BEST
    #1 score is worst
    # cosine distance is used to measure similarity between vectors. A score of 0 indicates that the vectors are identical, while a score closer to 1 indicates that they are less similar.
    # euclidean distance is another way to measure similarity between vectors. A score of 0 indicates that the vectors are identical, while a higher score indicates that they are less similar.

* [SIM=0.7338] Healthy Diet - # Healthy Living  
## Healthy Diet  
A healthy die
* [SIM=0.7280] Healthy Diet - ### Benefits  
A balanced diet can support overall
* [SIM=0.6732] Exercise - ## Exercise  
Regular physical activity can improv


In [31]:
pinecone_vector_store.delete(ids=['8746d3a2-a45b-4ee3-bb80-341cedb593a8'])

Weaviate - open source,supports hybrid search modular

In [14]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs=Document(markdown_texts)
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50,add_start_index=True)
chunks_rcts = splitter.split_documents([docs])


In [15]:
chunks_rcts

[Document(metadata={'start_index': 1}, page_content='# Healthy Living\n\n## Healthy Diet\n\nA healthy diet includes vegetables, fruits, whole grains,\nproteins, and sufficient water.\n\n### Benefits\n\nA balanced diet can support overall health and provide\nthe nutrients needed by the body.\n\n## Exercise'),
 Document(metadata={'start_index': 231}, page_content='## Exercise\n\nRegular physical activity can improve fitness and support\na healthy lifestyle.\n\n### Daily Activity\n\nWalking, cycling, swimming, and other physical activities\ncan be included in a daily routine.\n\n# Travel Tips and Adventure\n\n## Travel Planning'),
 Document(metadata={'start_index': 439}, page_content='# Travel Tips and Adventure\n\n## Travel Planning\n\nGood travel planning includes selecting a destination,\nchecking transportation, and preparing important documents.\n\n### Before Traveling\n\nCheck your passport, tickets, accommodation, weather,\nand local transportation before starting your journey.'),


In [16]:
import weaviate
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [17]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY")),
    headers={"X-Google-Api-Key": os.getenv("GOOGLE_API_KEY")}
)

In [23]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
pc = Pinecone(api_key=PINECONE_API_KEY)

In [28]:
from langchain_weaviate import WeaviateVectorStore
weaviate_vector_store = WeaviateVectorStore(
    client=weaviate_client,
    index_name="weaviate_tutorial",
    text_key="text",
    embedding=embeddings,
)

In [29]:
weaviate_vector_store.add_documents(chunks_rcts)

['3de6df34-c7a9-45f2-98f8-61a7f8a6bd9e',
 '3eac57d3-acc2-4de7-ba19-0096218613eb',
 '70d45cbe-8ded-4d51-8ef3-7c2dbbd1b06e',
 '4046f11e-60e6-479d-9f5a-61e1f809eebf']

In [30]:
results = weaviate_vector_store.similarity_search("healthy diet", k=3)
results

[Document(metadata={'start_index': 1.0}, page_content='# Healthy Living\n\n## Healthy Diet\n\nA healthy diet includes vegetables, fruits, whole grains,\nproteins, and sufficient water.\n\n### Benefits\n\nA balanced diet can support overall health and provide\nthe nutrients needed by the body.\n\n## Exercise'),
 Document(metadata={'start_index': 231.0}, page_content='## Exercise\n\nRegular physical activity can improve fitness and support\na healthy lifestyle.\n\n### Daily Activity\n\nWalking, cycling, swimming, and other physical activities\ncan be included in a daily routine.\n\n# Travel Tips and Adventure\n\n## Travel Planning'),
 Document(metadata={'start_index': 736.0}, page_content='## Safety Tips\n\nKeep important documents safe and stay aware of your\nsurroundings while traveling.\n\n### Adventure Travel\n\nAdventure activities can include hiking, camping,\ncycling, and exploring new places.')]

In [40]:
results = weaviate_vector_store.similarity_search_with_score(
    "healthy diet",
    k=3
)

for res, score in results:
    print(
        f"* [SIM={score:.4f}] "
        f"{res.metadata} "
        f"{res.page_content}"
    )

* [SIM=1.0000] {'start_index': 1.0} # Healthy Living

## Healthy Diet

A healthy diet includes vegetables, fruits, whole grains,
proteins, and sufficient water.

### Benefits

A balanced diet can support overall health and provide
the nutrients needed by the body.

## Exercise
* [SIM=0.2873] {'start_index': 231.0} ## Exercise

Regular physical activity can improve fitness and support
a healthy lifestyle.

### Daily Activity

Walking, cycling, swimming, and other physical activities
can be included in a daily routine.

# Travel Tips and Adventure

## Travel Planning
* [SIM=0.0004] {'start_index': 736.0} ## Safety Tips

Keep important documents safe and stay aware of your
surroundings while traveling.

### Adventure Travel

Adventure activities can include hiking, camping,
cycling, and exploring new places.


Milvus vdb - Open source, highly scalable

In [1]:
from langchain_milvus import Milvus
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [9]:
milvus_vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": os.getenv("MILVUS_ENDPOINT"), "token": os.getenv("MILVUS_TOKEN")},
    index_params={"index_type": "FLAT", "metric_type": "L2"},
    consistency_level="Strong",
    drop_old=False,
)

In [22]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(chunks_rcts))]

milvus_vector_store.add_documents(
    chunks_rcts,
    ids=uuids
)

['7feed6a2-30dd-44c1-8452-5e9d17bb6f57',
 'f095cd62-2bb8-4ce5-bbed-d7d65417f2cb',
 '68e4d672-98d0-403a-9ee1-15d62c31a2a4',
 'e663534d-17b8-40d7-adf6-4a95acf748d4']

In [23]:
uuids=[str(uuid4()) for _ in range(len(chunks_rcts))]
milvus_vector_store.add_documents(chunks_rcts, ids=uuids)

['505fe506-17d7-4d48-b06c-5c281fe12fa9',
 '4441c263-f4c0-4db8-ab6e-8db4f2e213d0',
 '13e5af86-d9b3-43d1-9e13-8d28780a4ff3',
 '0c12891d-add3-4fc1-85f9-c21c9c4929ef']

In [24]:
results = milvus_vector_store.similarity_search("what is a travel trip", k=3)
results

[Document(metadata={'start_index': 736, 'pk': '0c12891d-add3-4fc1-85f9-c21c9c4929ef'}, page_content='## Safety Tips\n\nKeep important documents safe and stay aware of your\nsurroundings while traveling.\n\n### Adventure Travel\n\nAdventure activities can include hiking, camping,\ncycling, and exploring new places.'),
 Document(metadata={'start_index': 736, 'pk': '9b531c04-9be4-4e4d-bf17-34588eaa8ba5'}, page_content='## Safety Tips\n\nKeep important documents safe and stay aware of your\nsurroundings while traveling.\n\n### Adventure Travel\n\nAdventure activities can include hiking, camping,\ncycling, and exploring new places.'),
 Document(metadata={'start_index': 736, 'pk': 'e663534d-17b8-40d7-adf6-4a95acf748d4'}, page_content='## Safety Tips\n\nKeep important documents safe and stay aware of your\nsurroundings while traveling.\n\n### Adventure Travel\n\nAdventure activities can include hiking, camping,\ncycling, and exploring new places.')]

In [25]:
results = milvus_vector_store.similarity_search_with_score("healthy diet", k=3)
for res, score in results:
    print(
        f"* [SIM={score:.4f}] "
        f"{res.metadata} "
        f"{res.page_content}"
    )

* [SIM=0.5614] {'start_index': 1, 'pk': '505fe506-17d7-4d48-b06c-5c281fe12fa9'} # Healthy Living

## Healthy Diet

A healthy diet includes vegetables, fruits, whole grains,
proteins, and sufficient water.

### Benefits

A balanced diet can support overall health and provide
the nutrients needed by the body.

## Exercise
* [SIM=0.5614] {'start_index': 1, 'pk': '7feed6a2-30dd-44c1-8452-5e9d17bb6f57'} # Healthy Living

## Healthy Diet

A healthy diet includes vegetables, fruits, whole grains,
proteins, and sufficient water.

### Benefits

A balanced diet can support overall health and provide
the nutrients needed by the body.

## Exercise
* [SIM=0.5614] {'start_index': 1, 'pk': 'af617568-5fb3-4a07-ac39-01b9a5df8ae1'} # Healthy Living

## Healthy Diet

A healthy diet includes vegetables, fruits, whole grains,
proteins, and sufficient water.

### Benefits

A balanced diet can support overall health and provide
the nutrients needed by the body.

## Exercise
